### Env and LLM initialisation

In [1]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import os

load = load_dotenv('./../.env', override=True)


# ollama_cloud_llm = ChatOllama(
#     base_url="http://localhost:11434/",  # Ollama cloud endpoint
#     model="devstral-small-2:24b-cloud", #gemini-3-flash-preview:cloud #qwen3.5:cloud
#     temperature=0.5,
#     max_tokens=1000,
#     headers={
#         "Authorization": f"Bearer {os.getenv('OLLAMA_CLOUD_API_KEY')}"  # Cloud auth
#     }
# )

ollama_local_llm = ChatOllama(
    base_url="http://localhost:11434/",
    model="llama3.2:latest",
    temperature=0.5,
    max_tokens=500,
    num_gpu=999
)

### Bringing back the code from pervious section for tool binding with LLM

In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

wikipidia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

search_tool = DuckDuckGoSearchRun()

@tool
def add_numbers(a: int, b:int) -> int:
    "Add two number and return results."
    return  int(a) + int(b)

@tool
def subtract_numbers(a: int, b:int) -> int:
    "Subtract two number and return results."
    return  int(a) - int(b)

@tool
def multiply_numbers(a: int, b:int) -> int:
    "Multiply two number and return results."
    return  int(a) * int(b)

tools = [wikipidia, add_numbers, subtract_numbers, multiply_numbers]

print(tools)


[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/home/shashank-sharma/Developer/Projects/langchain-trainings/myenv312/lib/python3.14/site-packages/wikipedia/__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)), StructuredTool(name='add_numbers', description='Add two number and return results.', args_schema=<class 'langchain_core.utils.pydantic.add_numbers'>, func=<function add_numbers at 0x7c709f4978a0>), StructuredTool(name='subtract_numbers', description='Subtract two number and return results.', args_schema=<class 'langchain_core.utils.pydantic.subtract_numbers'>, func=<function subtract_numbers at 0x7c709e9f5a60>), StructuredTool(name='multiply_numbers', description='Multiply two number and return results.', args_schema=<class 'langchain_core.utils.pydantic.multiply_numbers'>, func=<function multiply_numbers at 0x7c709e9f5bc0>)]


/home/shashank-sharma/Developer/Projects/langchain-trainings/myenv312/lib/python3.14/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### Agent code

In [28]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    tools=tools,
    model=ollama_local_llm,
    system_prompt="You are a helpful assistant that can answer questions using the available tools."
)

query = "What is the sum of 2 and 4? Also, Did donald trump won the 2024 presidential election and became president in 2025?"

result = agent.invoke ({"messages": [HumanMessage(content=query)]})

print(result)
print(result["messages"][-1].content)

{'messages': [HumanMessage(content='What is the sum of 2 and 4? Also, Did donald trump won the 2024 presidential election and became president in 2025?', additional_kwargs={}, response_metadata={}, id='e2cf4824-84b2-4438-8ac7-813c1b096f8a'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-05-11T12:31:04.406444036Z', 'done': True, 'done_reason': 'stop', 'total_duration': 476658318, 'load_duration': 86031291, 'prompt_eval_count': 353, 'prompt_eval_duration': 22821878, 'eval_count': 57, 'eval_duration': 341002691, 'logprobs': None, 'model_name': 'llama3.2:latest', 'model_provider': 'ollama'}, id='lc_run--019e1704-fef8-72b0-bc75-cdd1c7984214-0', tool_calls=[{'name': 'add_numbers', 'args': {'a': '2', 'b': '4'}, 'id': '6e2296d4-4bd8-4d02-ae59-ecee3d2fe939', 'type': 'tool_call'}, {'name': 'wikipedia', 'args': {'query': 'Did Donald Trump win the 2024 presidential election and become president in 2025?'}, 'id': '687a61ad-ea4a-4498-8

### Agent code with ChatPromptTemplate

In [8]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

agent = create_agent(
    tools=tools,
    model=ollama_local_llm,
    system_prompt="You are a helpful assistant who is actually expert in Maths and latest news. You can answer questions using the available tools."
)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant who is actually expert in Maths and latest news. You can answer questions using the available tools."),
    ("user", "What is the sum of 2 and 3?"),
    ("user", "What is latest movie of Tom Cruise hitting theatter in 2025?"),
    ("user", "Give me both the ansewers in JSON format")
])

result = agent.invoke({"messages": prompt_template.format_messages()})

print(result)
print(result["messages"][-1].content)

{'messages': [SystemMessage(content='You are a helpful assistant who is actually expert in Maths and latest news. You can answer questions using the available tools.', additional_kwargs={}, response_metadata={}, id='f50f3b5e-db4f-4650-889f-4335e7869a64'), HumanMessage(content='What is the sum of 2 and 3?', additional_kwargs={}, response_metadata={}, id='194e5d95-b265-4326-920a-f568467c1c68'), HumanMessage(content='What is latest movie of Tom Cruise hitting theatter in 2025?', additional_kwargs={}, response_metadata={}, id='a74eb17a-0f73-4bd7-9408-de40785c632b'), HumanMessage(content='Give me both the ansewers in JSON format', additional_kwargs={}, response_metadata={}, id='bbb8153e-e22c-4b5d-bf51-e6872d7085fb'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-05-12T06:38:32.157905734Z', 'done': True, 'done_reason': 'stop', 'total_duration': 463216386, 'load_duration': 80165938, 'prompt_eval_count': 391, 'prompt_eval_duratio

### Using Playwright Browser Toolkit

In [ ]:
#pip install -qU  playwright
#pip install -qU  lxml

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import (
    create_async_playwright_browser,
)
import nest_asyncio

nest_asyncio.apply()



### Instantiating a browser toolkit

In [4]:
async_browser = create_async_playwright_browser()
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()
tools

[ClickTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>),
 NavigateTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>),
 NavigateBackTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>),
 ExtractTextTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>),
 ExtractHyperlinksTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>),
 GetElementsTool(async_browser=<Brows

In [5]:
tools_by_name = {tool.name: tool for tool in tools}
navigate_tool = tools_by_name["navigate_browser"]
get_element_tool = tools_by_name["get_elements"]
navigate_tool, get_element_tool

(NavigateTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>),
 GetElementsTool(async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/shashank-sharma/.cache/ms-playwright/chromium-1217/chrome-linux64/chrome> version=147.0.7727.15>))

In [6]:
await navigate_tool.arun({"url": "http://eaapp.somee.com/Employee/"})

'Navigating to http://eaapp.somee.com/Employee/ returned status code 200'

In [9]:
await get_element_tool.arun({
    "selector": "td",
    "action": "innerText"
})

'[{"innerText": "Małgorzata Kędzierska"}, {"innerText": "63 yrs\\n🔴 Retirement Age"}, {"innerText": "$8,000.00"}, {"innerText": "18 mo."}, {"innerText": "Middle"}, {"innerText": "malgorzata.kedzierska@company.com"}, {"innerText": "📋 Details"}, {"innerText": "John Anderson"}, {"innerText": "32 yrs"}, {"innerText": "$5,500.00"}, {"innerText": "15 mo."}, {"innerText": "Middle"}, {"innerText": "john.anderson@example.com"}, {"innerText": "📋 Details"}, {"innerText": "John Smith"}, {"innerText": "30 yrs"}, {"innerText": "$50,000.00"}, {"innerText": "12 mo."}, {"innerText": "Middle"}, {"innerText": "john.smith.updated@company.com"}, {"innerText": "📋 Details"}, {"innerText": "Michael Johnson"}, {"innerText": "28 yrs"}, {"innerText": "$4,500.00"}, {"innerText": "12 mo."}, {"innerText": "Junior"}, {"innerText": "michael.johnson@example.com"}, {"innerText": "📋 Details"}, {"innerText": "John Doe"}, {"innerText": "30 yrs"}, {"innerText": "$5,000.00"}, {"innerText": "12 mo."}, {"innerText": "Middle"}